### Блокнот чисто для тестов со Spark. **Dev-only**

In [2]:
try: spark.stop() # type: ignore
except: pass

from pyspark.sql import SparkSession

PACKAGES = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
])

spark = (SparkSession.builder
    .appName("dev")
    .master("local[*]")
    .config("spark.jars.packages", PACKAGES)
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cf6d78af-31fc-4709-94da-7084ce14f450;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.5 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.5 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#c

In [3]:
# Batch-чтение (НЕ readStream) — прочитает текущее содержимое и закроется
df = (spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "crypto.tickers")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load())

df.selectExpr("CAST(value AS STRING) as json").show(10, truncate=False)
print(f"Total messages: {df.count()}")

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|json                                                                                                                                                                                                                                                                                                                                                      |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

[Stage 1:=========>                                                 (1 + 5) / 6]

Total messages: 12374


In [4]:
print(spark.sparkContext.getConf().get("spark.jars"))
print(spark.sparkContext.getConf().get("spark.jars.packages"))

file:///root/.ivy2/jars/org.apache.spark_spark-sql-kafka-0-10_2.12-3.5.5.jar,file:///root/.ivy2/jars/org.apache.hadoop_hadoop-aws-3.3.4.jar,file:///root/.ivy2/jars/com.amazonaws_aws-java-sdk-bundle-1.12.262.jar,file:///root/.ivy2/jars/org.apache.spark_spark-token-provider-kafka-0-10_2.12-3.5.5.jar,file:///root/.ivy2/jars/org.apache.kafka_kafka-clients-3.4.1.jar,file:///root/.ivy2/jars/com.google.code.findbugs_jsr305-3.0.0.jar,file:///root/.ivy2/jars/org.apache.commons_commons-pool2-2.11.1.jar,file:///root/.ivy2/jars/org.apache.hadoop_hadoop-client-runtime-3.3.4.jar,file:///root/.ivy2/jars/org.lz4_lz4-java-1.8.0.jar,file:///root/.ivy2/jars/org.xerial.snappy_snappy-java-1.1.10.5.jar,file:///root/.ivy2/jars/org.slf4j_slf4j-api-2.0.7.jar,file:///root/.ivy2/jars/org.apache.hadoop_hadoop-client-api-3.3.4.jar,file:///root/.ivy2/jars/commons-logging_commons-logging-1.1.3.jar,file:///root/.ivy2/jars/org.wildfly.openssl_wildfly-openssl-1.0.7.Final.jar
org.apache.spark:spark-sql-kafka-0-10_2.12:3

In [5]:
df.select("*").limit(10).show()

+----+--------------------+--------------+---------+------+--------------------+-------------+
| key|               value|         topic|partition|offset|           timestamp|timestampType|
+----+--------------------+--------------+---------+------+--------------------+-------------+
|NULL|[7B 22 74 6F 70 6...|crypto.tickers|        3|     0|2026-04-29 14:54:...|            0|
|NULL|[7B 22 74 6F 70 6...|crypto.tickers|        3|     1|2026-04-29 14:54:...|            0|
|NULL|[7B 22 74 6F 70 6...|crypto.tickers|        3|     2|2026-04-29 14:54:...|            0|
|NULL|[7B 22 74 6F 70 6...|crypto.tickers|        3|     3|2026-04-29 14:54:...|            0|
|NULL|[7B 22 74 6F 70 6...|crypto.tickers|        3|     4|2026-04-29 14:54:...|            0|
|NULL|[7B 22 74 6F 70 6...|crypto.tickers|        3|     5|2026-04-29 14:54:...|            0|
|NULL|[7B 22 74 6F 70 6...|crypto.tickers|        3|     6|2026-04-29 14:54:...|            0|
|NULL|[7B 22 74 6F 70 6...|crypto.tickers|        

In [4]:
import pyspark.sql.functions as F

TOPIC = "crypto.tickers"
CHECKPOINT_PATH = f"s3a://spark-checkpoints/bronze{TOPIC}"
BRONZE_PATH = f"s3a://crypto-lake/bronze/{TOPIC}"

source = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
         )

parsed = (source.selectExpr("CAST(value AS STRING) as raw_json", "topic", "partition", "offset", "timestamp as kafka_ts")
     .withColumn("ingestion_ts", F.current_timestamp())
     .withColumn("year",  F.year("kafka_ts"))
     .withColumn("month", F.month("kafka_ts"))
     .withColumn("day",   F.dayofmonth("kafka_ts"))
     .withColumn("hour",  F.hour("kafka_ts")))

'''write = (parsed.writeStream.format("parquet")
        .option("path", BRONZE_PATH)
        .option("checkpointLocation", CHECKPOINT_PATH)
        .partitionBy("year", "month", "day", "hour")
        .trigger(availableNow=True)
        .outputMode("append")
        .start())'''

write = (parsed.writeStream
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="60 seconds")
    .start())
write.awaitTermination()


-------------------------------------------
Batch: 0
-------------------------------------------
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+---------+------+-----------------------+-----------------------+----+-----+---+----+
|raw_json                                                                                                                                                                                                                                                                                                                                                  |topic         |partition|offset|kafka_ts               |ingestion_ts           |year|month|day|hour|
+----

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

-------------------------------------------
Batch: 1
-------------------------------------------
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+---------+------+-----------------------+-----------------------+----+-----+---+----+
|raw_json                                                                                                                                                                                                                                                                                                                                                  |topic         |partition|offset|kafka_ts               |ingestion_ts           |year|month|day|hour|
+----

-------------------------------------------
Batch: 7
-------------------------------------------
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+---------+------+-----------------------+-----------------------+----+-----+---+----+
|raw_json                                                                                                                                                                                                                                                                                                                                                  |topic         |partition|offset|kafka_ts               |ingestion_ts           |year|month|day|hour|
+----

-------------------------------------------
Batch: 14
-------------------------------------------
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+---------+------+-----------------------+-----------------------+----+-----+---+----+
|raw_json                                                                                                                                                                                                                                                                                                                                                  |topic         |partition|offset|kafka_ts               |ingestion_ts           |year|month|day|hour|
+---

-------------------------------------------
Batch: 15
-------------------------------------------
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+---------+------+-----------------------+-----------------------+----+-----+---+----+
|raw_json                                                                                                                                                                                                                                                                                                                                                  |topic         |partition|offset|kafka_ts               |ingestion_ts           |year|month|day|hour|
+---

In [ ]:
import pyspark.sql.functions as F

TOPIC = "crypto.prices"
CHECKPOINT_PATH = f"s3a://spark-checkpoints/bronze{TOPIC}"
BRONZE_PATH = f"s3a://crypto-lake/bronze/{TOPIC}"

source = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
         )

parsed = (source.selectExpr("CAST(value AS STRING) as raw_json", "topic", "partition", "offset", "timestamp as kafka_ts")
     .withColumn("ingestion_ts", F.current_timestamp())
     .withColumn("year",  F.year("kafka_ts"))
     .withColumn("month", F.month("kafka_ts"))
     .withColumn("day",   F.dayofmonth("kafka_ts"))
     .withColumn("hour",  F.hour("kafka_ts")))

'''write = (parsed.writeStream.format("parquet")
        .option("path", BRONZE_PATH)
        .option("checkpointLocation", CHECKPOINT_PATH)
        .partitionBy("year", "month", "day", "hour")
        .trigger(availableNow=True)
        .outputMode("append")
        .start())'''

write = (clean.writeStream
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="60 seconds")
    .start())
write.awaitTermination()


write.awaitTermination()